Import important packages

In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


Loading data and changing missing data values to Null

In [53]:
df = pd.read_csv("data/train.csv")
# Remove ? with NULL
df = df.replace(r"^\s*\?\s*$", pd.NA, regex=True)
# Remove Unkown/Invalid with NULL
df = df.replace(r"^\s*Unknown/Invalid\s*$", pd.NA, regex=True)
# Remove empty values with NULL
df = df.replace(r"^\s*$", pd.NA, regex=True)

#----------------------------------
#Setting unkown values from IDS_mapping to NULL in train.csv

# From IDS_mapping.csv
ids_to_null = {
    "admission_type_id": [5, 6, 8],                 # Not Available, NULL, Not Mapped
    "discharge_disposition_id": [18, 25, 26],       # NULL, Not Mapped, Unknown/Invalid
    "admission_source_id": [9, 15, 17, 20, 21],     # Not Available, NULL, Not Mapped, Unknown/Invalid
}

for col, bad_codes in ids_to_null.items():
    df[col] = df[col].replace(bad_codes, pd.NA)



checking that data is gone

In [54]:
# Check that mapped ID placeholders are gone
print((df["admission_type_id"].isin([5,6,8])).sum())
print((df["discharge_disposition_id"].isin([18,25,26])).sum())
print((df["admission_source_id"].isin([9,15,17,20,21])).sum())

# Check key placeholder strings are gone
for c in ["weight","payer_code","medical_specialty","race","gender","diag_1","diag_2","diag_3"]:
    print(c, (df[c].astype(str).str.strip() == "?").sum(), (df[c].astype(str).str.strip() == "Unknown/Invalid").sum())


0
0
0
weight 0 0
payer_code 0 0
medical_specialty 0 0
race 0 0
gender 0 0
diag_1 0 0
diag_2 0 0
diag_3 0 0


checking information on featre

In [55]:

df.shape
df.head()
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 71236 entries, 0 to 71235
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        71236 non-null  int64 
 1   encounter_id              71236 non-null  int64 
 2   patient_nbr               71236 non-null  int64 
 3   race                      69615 non-null  str   
 4   gender                    71233 non-null  str   
 5   age                       71236 non-null  str   
 6   weight                    2250 non-null   str   
 7   admission_type_id         64013 non-null  object
 8   discharge_disposition_id  67917 non-null  object
 9   admission_source_id       66359 non-null  object
 10  time_in_hospital          71236 non-null  int64 
 11  payer_code                43058 non-null  str   
 12  medical_specialty         36306 non-null  str   
 13  num_lab_procedures        71236 non-null  int64 
 14  num_procedures            71236 n

,id,encounter_id,patient_nbr,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,71236.000000,7.123600e+04,7.123600e+04,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000
mean,50874.217250,1.651772e+08,5.421577e+07,4.394660,43.112612,1.342523,16.016452,0.371104,0.195800,0.635353,7.418103
std,29423.014266,1.027123e+08,3.866893e+07,2.979942,19.658896,1.705587,8.130801,1.269937,0.864571,1.269287,1.930480
min,2.000000,1.573800e+04,1.350000e+02,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,25319.750000,8.471072e+07,2.340213e+07,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,50867.500000,1.523227e+08,4.520375e+07,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,76450.250000,2.307948e+08,8.742009e+07,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,101766.000000,4.438672e+08,1.895026e+08,14.000000,132.000000,6.000000,81.000000,42.000000,63.000000,19.000000,16.000000


Checking different features unique values

In [ ]:
# Dropping unnecisary features
df = df.drop(columns = ["id",
         "encounter_id",
         "patient_nbr",
         "payer_code",
         ], errors="ignore")

# Dropping features with to many missing values
df = df.drop(columns = [
        "weight",
        "max_glu_serum",
        "A1Cresult",
        "medical_specialty",
        ], errors="ignore")

# Dropping features with no variation
df = df.drop(columns = [
        "troglitazone",
        "examide",
        "citoglipton",
         ], errors="ignore")

removing unsuable features

In [59]:

# print Missing % in each feature
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print ("Missing % of data in features")
print(missing_pct.head(20))

# print Low-variance (few unique values)
nunique = df.nunique(dropna=False).sort_values()
print(nunique.head(20))


Missing % of data in features
admission_type_id           10.139536
admission_source_id          6.846258
discharge_disposition_id     4.659161
race                         2.275535
diag_3                       1.388343
diag_2                       0.342523
diag_1                       0.021057
gender                       0.004211
num_lab_procedures           0.000000
time_in_hospital             0.000000
age                          0.000000
num_procedures               0.000000
number_emergency             0.000000
number_outpatient            0.000000
num_medications              0.000000
number_inpatient             0.000000
number_diagnoses             0.000000
metformin                    0.000000
repaglinide                  0.000000
nateglinide                  0.000000
dtype: float64
tolbutamide                 2
acetohexamide               2
change                      2
diabetesMed                 2
metformin-pioglitazone      2
metformin-rosiglitazone     2
glimepiride-pio